# Gas Turbine · отложенный тест для NOx и CO
В обзоре одна пара seed дала меньшую валидационную RMSE у MGS-KAN на NOx и у MLP на CO. Проверяем обе заранее выбранные конфигурации на девяти общих входах. Обучение — 2011–2012 годы, выбор эпохи по минимуму RMSE — 2013 год. Тест объединяет все наблюдения 2014 и 2015 годов.

| Волна | MGS-KAN (ядер на признак / пару) | Параметры MGS-KAN / MLP |
|---|---|---|
| 1 | 7 / 4 | 1 063 / 1 054 |
| 2 | 16 / 8 | 2 170 / 2 174 |

MLP содержит два скрытых ReLU-слоя одинаковой ширины. Для каждой цели и волны сохраняем 100 эпох Adam, шаг 0,001 и батч 64. Сначала восстановим восемь чекпоинтов исходного обзора. Добавим две пары seed и оценим все 24 модели на одном временном тесте. Между seed меняются инициализация и порядок перемешивания. Разброс отражает влияние этих двух факторов на фиксированном разбиении.

In [ ]:
PROJECT_DIR = None
DEVICE = "cuda:1"  # Свободная GPU в Jupyter для новых обучений.
VERSION = "gas-turbine-confirm-v1"
GPU_BUDGET_MIB = 4096

In [ ]:
import sys
from pathlib import Path

if PROJECT_DIR is None:
    cwd = Path.cwd().resolve()
    checkout = next(
        (folder for folder in (cwd, *cwd.parents) if (folder / "src/experiments/runner.py").is_file()),
        None,
    )
else:
    checkout = Path(PROJECT_DIR).expanduser()
    if not checkout.is_absolute() or not (checkout / "src/experiments/runner.py").is_file():
        raise ValueError("Укажи в PROJECT_DIR абсолютный путь к корню mgs-kan на сервере.")
    checkout = checkout.resolve()

if checkout is not None and sys.path[0:1] != [str(checkout)]:
    sys.path.insert(0, str(checkout))

try:
    from src.paths import DATA_DIR, ENV_FILE, PROJECT_ROOT
except ModuleNotFoundError as error:
    raise ImportError("Укажи PROJECT_DIR и выполни ячейки заново.") from error

if checkout is not None and PROJECT_ROOT != checkout:
    raise RuntimeError("Ядро уже загрузило src из другой папки. Перезапусти ядро.")
if not (PROJECT_ROOT / "notebooks/baselines/run_series.ipynb").is_file():
    raise RuntimeError("Укажи PROJECT_DIR с рабочей копией mgs-kan и перезапусти ядро.")
print(f"Проект: {PROJECT_ROOT}\nНастройки: {ENV_FILE}\nДанные: {DATA_DIR}")

In [ ]:
import math
from itertools import product

import pandas as pd
import torch
from IPython.display import display

from src.config import MonitorSettings, ResourceSettings, TrackingSettings
from src.data import load_dataset
from src.experiments import ExperimentSeries, Trial
from src.experiments.search import ValidationGrid
from src.models import make_model, parameter_count
from src.tracking import MlflowTracker
from src.tracking.recipe import ValidationRecipe
from src.training import Trainer

targets = ("gas_turbine_nox", "gas_turbine_co")
models = ("mgs_kan", "mlp")
original_seed = (11, 101)
new_seeds = ((22, 202), (33, 303))
survey = ValidationGrid.preset("benchmark_survey", datasets=targets)
stages = tuple(s.model_copy(update={"models": models}) for s in survey.stages)
grid = ValidationGrid.model_validate({**survey.model_dump(), "stages": stages, "seeds": new_seeds})
tracking = TrackingSettings(_env_file=ENV_FILE)
tracker = MlflowTracker(tracking)
resources = ResourceSettings(_env_file=ENV_FILE, max_parallel_trials=1, cpu_threads=1,
                             gpu_budget_mib=GPU_BUDGET_MIB)
monitoring = MonitorSettings(_env_file=ENV_FILE)
output = PROJECT_ROOT / "results" / "gas-turbine-heldout" / VERSION
data = {target: load_dataset(target, root=DATA_DIR, fold=0) for target in targets}

series = {}
expected_sizes = {"survey-wave-1": (1063, 1054), "survey-wave-2": (2170, 2174)}
for target, _, columns, stage, settings in grid.plan():
    opts = settings.model_copy(update={"features": data[target].n_features, "init_seed": 11})
    if tuple(parameter_count(make_model(m, opts)) for m in models) != expected_sizes[stage.name]:
        raise ValueError(f"Изменились размеры моделей для {target} / {stage.name}")
    series[target, stage.name] = ExperimentSeries(
        series_id=stage.name, dataset=target, models=models, folds=(0,), feature_columns=columns,
        seeds=(original_seed, *new_seeds), device=DEVICE, evaluation="validation",
        training=grid.training, model_settings=settings, resources=resources,
    )
print(f"8 чекпоинтов из обзора, до {grid.size} новых обучений · {DEVICE}")

## Исходные чекпоинты
Идентификаторы взяты из выгрузок двух волн. В MLflow проверяются целевая переменная, хеш данных, порядок входов, seed, параметры обучения и архитектура. После загрузки весов на CPU сверяем номер эпохи и повторно вычисляем RMSE на 2013 году. Это позволяет обнаружить несовпадение до новых обучений.

In [ ]:
survey_ids = {
    ("gas_turbine_co", "survey-wave-1", "mlp"): "0bead4c91f89484e928fd56877483eab",
    ("gas_turbine_co", "survey-wave-1", "mgs_kan"): "27e16345f63c4d1da5145f1edacfab21",
    ("gas_turbine_nox", "survey-wave-1", "mlp"): "e0eb5cb0482a4bf0bfe244ee0a9c6ec4",
    ("gas_turbine_nox", "survey-wave-1", "mgs_kan"): "3adc5ca916ea48afb7bcbefd350cd2df",
    ("gas_turbine_co", "survey-wave-2", "mlp"): "62e86f648952495cb6ca2060604c105d",
    ("gas_turbine_co", "survey-wave-2", "mgs_kan"): "c305e72645e14a1bb402624f2d5483bd",
    ("gas_turbine_nox", "survey-wave-2", "mlp"): "c024275cd88541f0a173afb97a45f8c3",
    ("gas_turbine_nox", "survey-wave-2", "mgs_kan"): "0bcd23bf9e8f41b8ae26ca79857010fb",
}

In [ ]:
def scores(trainer, dataset, partition):
    features, targets_scaled = partition.tensors
    batches = features.split(grid.training.batch_size)
    pred_scaled = torch.cat([trainer.predict(x).cpu() for x in batches])
    pred = dataset.inverse_target(pred_scaled.double())
    actual = dataset.inverse_target(targets_scaled.double())
    sse = (pred - actual).square().sum().item()
    sst = (actual - actual.mean()).square().sum().item()
    if not math.isfinite(sse) or sst <= 0:
        raise ValueError("Проверь прогнозы и разброс целевой переменной")
    return math.sqrt(sse / len(actual)), 1 - sse / sst


def restore(target, stage, model, seeds, run_id):
    dataset = data[target]
    config = series[target, stage]
    trial = Trial(model=model, fold=0, init_seed=seeds[0], shuffle_seed=seeds[1])
    recipe = ValidationRecipe.for_trial(config, trial, dataset)
    run = tracker.client.get_run(run_id)
    if (run.info.status != "FINISHED" or run.data.tags.get("mgs_kan.evaluation") != "validation"
            or not recipe.matches(run.data.params)):
        raise ValueError(f"{run_id}: запуск не соответствует протоколу")
    opts = config.model_settings.model_copy(update={"features": dataset.n_features, "init_seed": seeds[0]})
    trainer = Trainer(make_model(model, opts), tracker=tracker)
    epoch = trainer.load_checkpoint(run_id=run_id)
    rmse, _ = scores(trainer, dataset, dataset.val)
    if epoch != run.data.metrics["best_epoch"] or not math.isclose(
        rmse, run.data.metrics["best_val_rmse"], rel_tol=1e-4, abs_tol=1e-4,
    ):
        raise ValueError(f"{run_id}: эпоха или RMSE чекпоинта расходится с MLflow")
    row = dict(dataset=target, stage=stage, model=model, init_seed=seeds[0], shuffle_seed=seeds[1],
               parameters=parameter_count(trainer.model), run_id=run_id, best_epoch=epoch, val_rmse=rmse)
    return row, trainer

In [ ]:
if set(survey_ids) != set(product(targets, expected_sizes, models)) or len(set(survey_ids.values())) != 8:
    raise ValueError("Нужны восемь уникальных запусков исходного обзора")
original = []
for (target, stage, model), run_id in survey_ids.items():
    row, trainer = restore(target, stage, model, original_seed, run_id)
    original.append((dict(row, source="survey"), trainer))
print(f"Проверено чекпоинтов обзора: {len(original)}")

## Две дополнительные пары seed
Сетка запрашивает только (22, 202) и (33, 303). Если часть комбинаций уже завершена, MLflow вернёт подходящие валидационные запуски с чекпоинтами. После прерывания повтори ячейку с той же конфигурацией.

In [ ]:
additional = grid.run(version=VERSION, device=DEVICE, root=DATA_DIR, resources=resources,
                      tracking=tracking, monitoring=monitoring)
print(f"Дополнительные комбинации: {len(additional)}, из них из MLflow: {additional.reused.sum()}")

In [ ]:
verified = list(original)
for result in additional.itertuples():
    seeds = (result.init_seed, result.shuffle_seed)
    row, trainer = restore(result.dataset, result.stage, result.model, seeds, result.run_id)
    verified.append((dict(row, source="cache" if result.reused else "new"), trainer))

keys = ["dataset", "stage", "init_seed", "shuffle_seed"]
planned = {(t, w, i, s, m) for t, w, (i, s), m in
           product(targets, expected_sizes, (original_seed, *new_seeds), models)}
manifest = pd.DataFrame(row for row, _ in verified)
observed = set(manifest[keys + ["model"]].itertuples(index=False, name=None))
if observed != planned or len(manifest) != 24 or manifest.run_id.nunique() != 24:
    raise ValueError("Нужны все 24 уникальных чекпоинта до оценки на тесте")

## Парное сравнение на тесте 2014–2015
Для каждого чекпоинта считаем RMSE и R² в исходной шкале цели. Сопоставляем модели внутри одной цели, волны и пары seed. Отрицательная разность RMSE означает выигрыш MGS-KAN. В сводке — средние по трём парам seed и число выигрышей MGS-KAN из трёх. Обе волны входят в результат независимо от порядка на тесте.

In [ ]:
rows = []
for row, trainer in verified:
    dataset = data[row["dataset"]]
    rmse, r2 = scores(trainer, dataset, dataset.test)
    rows.append(dict(row, n_test=len(dataset.test), test_rmse=rmse, test_r2=r2))
runs = pd.DataFrame(rows)
paired = runs.pivot(index=keys, columns="model", values="test_rmse").rename_axis(columns=None)
paired["delta_rmse"] = paired["mgs_kan"] - paired["mlp"]
paired["mgs_kan_win"] = paired["delta_rmse"] < 0
summary = paired.groupby(["dataset", "stage"]).agg(
    mgs_kan_rmse=("mgs_kan", "mean"), mlp_rmse=("mlp", "mean"),
    mean_delta=("delta_rmse", "mean"), sd_delta=("delta_rmse", "std"),
    mgs_kan_wins=("mgs_kan_win", "sum"), seed_pairs=("delta_rmse", "size"),
)
output.mkdir(parents=True, exist_ok=True)
runs.to_csv(output / "runs.csv", index=False)
paired.to_csv(output / "paired.csv")
summary.to_csv(output / "summary.csv")
display(summary.round(4))
display(paired.round(4))
print(f"Результаты: {output}")